In [16]:
import sys
import json
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from openbabel import openbabel

In [2]:

def convert_smiles_to_3d_pdb(smiles, output_filename):
    """
    根据论文描述将 SMILES 转换为 3D PDB 结构：
    1. 使用 RDKit 解析 SMILES 
    2. 添加显式氢 (Add explicit hydrogens) 
    3. 生成初始 3D 构象
    4. 使用 MMFF94 力场进行能量最小化 (Energy minimization using MMFF94) 
    5. 保存为 PDB 文件
    """
    try:
        # 1. 从 SMILES 创建分子对象
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"Error: 无法解析 SMILES: {smiles}")
            return

        # 2. 添加显式氢原子
        # 论文原文: "Explicit hydrogens were added" 
        mol_h = Chem.AddHs(mol)

        # 3. 生成初始 3D 坐标 (Embedding)
        # 这是进行能量最小化的前提，需要先有一个粗略的 3D 构象
        embed_params = AllChem.ETKDG()
        embed_status = AllChem.EmbedMolecule(mol_h, params=embed_params)
        
        if embed_status != 0:
            print(f"Error: 无法生成 3D 构象: {smiles}")
            # 尝试使用随机坐标作为备选方案
            embed_status = AllChem.EmbedMolecule(mol_h, useRandomCoords=True)
            if embed_status != 0:
                return

        # 4. 使用 MMFF94 力场进行能量最小化
        # 论文原文: "...structures were subjected to a short energy minimization using the MMFF94 force field" 
        try:
            # 检查 MMFF94 是否可用
            if AllChem.MMFFHasAllMoleculeParams(mol_h):
                AllChem.MMFFOptimizeMolecule(mol_h, mmffVariant='MMFF94')
            else:
                print("Warning: MMFF94 参数不全，尝试使用 UFF 作为替代，或者跳过优化。")
                # 严格按照论文应使用 MMFF94，如果失败可能需要检查分子类型
        except Exception as e:
            print(f"Optimization error: {e}")

        # 5. 保存为 PDB 文件
        # 虽然论文提到最终生成了对接用的 SDF [cite: 288]，但也经常使用 PDB 格式保存复合物结构
        Chem.MolToPDBFile(mol_h, output_filename)
        print(f"Success: 已保存 3D 结构到 {output_filename}")

    except Exception as e:
        print(f"发生未预期的错误: {e}")


In [10]:

def generate_skid_style_pdb(smiles, output_filename="ligand.pdb"):
    """
    结合 SKiD GitHub 仓库逻辑：
    1. 使用 RDKit 进行 3D 构象生成和 MMFF94 能量优化。
    2. (可选) 使用 OpenBabel 进行最终格式标准化（如果需要）。
    """
    try:
        # --- 第一阶段：使用 RDKit 生成高质量 3D 结构 ---
        
        # 1. 解析 SMILES
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            raise ValueError(f"Invalid SMILES: {smiles}")

        # 2. 添加显式氢 (论文核心步骤)
        mol = Chem.AddHs(mol)

        # 3. 生成 3D 坐标 (Embedding)
        # ETKDG 是目前最标准的生成算法
        params = AllChem.ETKDG()
        params.randomSeed = 0xf00d  # 设置随机种子以保证结果可复现
        embed_res = AllChem.EmbedMolecule(mol, params)
        
        if embed_res == -1:
            # 如果失败，尝试随机坐标
            AllChem.EmbedMolecule(mol, useRandomCoords=True)

        # 4. MMFF94 力场能量最小化 (论文核心步骤)
        # SKiD 使用此步骤确保分子构象符合物理化学规律
        if AllChem.MMFFHasAllMoleculeParams(mol):
            AllChem.MMFFOptimizeMolecule(mol, mmffVariant='MMFF94')
        else:
            # 如果缺少 MMFF94 参数，回退到 UFF 力场 (常见处理逻辑)
            AllChem.UFFOptimizeMolecule(mol)

        # --- 第二阶段：输出 PDB ---
        
        # 方法 A: 直接使用 RDKit 输出 (最简单)
        Chem.MolToPDBFile(mol, output_filename)
        
        # 方法 B: (高级) 如果 RDKit 输出的 PDB 在后续对接软件(GNINA)中兼容性不佳，
        # SKiD 可能会调用 OpenBabel 进行转换。以下演示如何用 OpenBabel 转换：
        # (通常在命令行中运行: obabel -isdf input.sdf -opdb -O output.pdb)
        
        print(f"成功生成文件: {output_filename}")

    except Exception as e:
        print(f"Error processing {smiles}: {e}")


In [12]:

sample_smiles = "CC1CC23CCC4C(C)(C(O)OC5OC(CO)C(O)C(O)C5O)CCCC4(C)C2CCC1(OC1OC(CO)C(O)C(OC2OC(CO)C(O)C(O)C2O)C1OC1OC(CO)C(O)C(O)C1O)C3"
output_file = r"E:\pUGTdb\experiment_data\temp_RA-bcc_dock001.pdb"

# convert_smiles_to_3d_pdb(sample_smiles, output_file)
generate_skid_style_pdb(sample_smiles, output_file)

成功生成文件: E:\pUGTdb\experiment_data\temp_RA-bcc_dock001.pdb


In [19]:
with open(r'E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\smiles_to_id.json', 'r') as f:
    smiles = json.load(f)

In [25]:
for k,v in smiles.items():
    output_file = rf'E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\{v}.pdb'
    generate_skid_style_pdb(k, output_file)

[21:08:05] UFFTYPER: Unrecognized atom type: *_ (0)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (12)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (13)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (14)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (15)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (16)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (18)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (24)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (25)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (0)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (12)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (13)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (14)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (15)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (16)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (18)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (24)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (25)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ 

成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00001.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00002.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00003.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00004.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00005.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00006.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00007.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00008.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_P

[21:08:05] UFFTYPER: Unrecognized atom type: *_ (18)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (20)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (18)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (20)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (18)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (20)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (18)
[21:08:05] UFFTYPER: Unrecognized atom type: *_ (20)


成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00011.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00012.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00013.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00014.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00015.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00016.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00017.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00018.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_P

[21:08:06] UFFTYPER: Unrecognized atom type: *_ (10)
[21:08:06] UFFTYPER: Unrecognized atom type: *_ (17)
[21:08:06] UFFTYPER: Unrecognized atom type: *_ (18)
[21:08:06] UFFTYPER: Unrecognized atom type: *_ (19)
[21:08:06] UFFTYPER: Unrecognized atom type: *_ (21)
[21:08:06] UFFTYPER: Unrecognized atom type: *_ (22)
[21:08:06] UFFTYPER: Unrecognized atom type: *_ (10)
[21:08:06] UFFTYPER: Unrecognized atom type: *_ (17)
[21:08:06] UFFTYPER: Unrecognized atom type: *_ (18)
[21:08:06] UFFTYPER: Unrecognized atom type: *_ (19)
[21:08:06] UFFTYPER: Unrecognized atom type: *_ (21)
[21:08:06] UFFTYPER: Unrecognized atom type: *_ (22)


...

成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00052.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00053.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00054.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00055.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00056.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00057.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00058.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00059.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_P

[21:08:20] UFFTYPER: Unrecognized atom type: *_ (6)
[21:08:20] UFFTYPER: Unrecognized atom type: *_ (8)
[21:08:20] UFFTYPER: Unrecognized atom type: *_ (10)
[21:08:20] UFFTYPER: Unrecognized atom type: *_ (12)
[21:08:20] UFFTYPER: Unrecognized atom type: *_ (18)
[21:08:20] UFFTYPER: Unrecognized atom type: *_ (20)
[21:08:20] UFFTYPER: Unrecognized atom type: *_ (22)
[21:08:20] UFFTYPER: Unrecognized atom type: *_ (24)
[21:08:20] UFFTYPER: Unrecognized atom type: *_ (26)
[21:08:20] UFFTYPER: Unrecognized atom type: *_ (6)
[21:08:20] UFFTYPER: Unrecognized atom type: *_ (8)
[21:08:20] UFFTYPER: Unrecognized atom type: *_ (10)
[21:08:20] UFFTYPER: Unrecognized atom type: *_ (12)
[21:08:20] UFFTYPER: Unrecognized atom type: *_ (18)
[21:08:20] UFFTYPER: Unrecognized atom type: *_ (20)
[21:08:20] UFFTYPER: Unrecognized atom type: *_ (22)
[21:08:20] UFFTYPER: Unrecognized atom type: *_ (24)
[21:08:20] UFFTYPER: Unrecognized atom type: *_ (26)


成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00081.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00082.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00083.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00084.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00085.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00086.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00087.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00088.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_P

[21:08:24] WARNING: not removing hydrogen atom without neighbors
[21:08:24] WARNING: not removing hydrogen atom without neighbors


成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00091.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00092.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00093.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00094.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00095.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00096.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00097.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\SMILES_00098.pdb
成功生成文件: E:\pUGTdb\20251128_collection\download_P

# fhdaoifhsdao;fh;sda

In [2]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10,10))
plt.imshow(plt.imread(r'E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\RDKit_Donor\PF00201_taxonomy33090_1.pdb.png'))
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'E:\\pUGTdb\\20251128_collection\\download_PF00201_taxonomy33090\\step2_get_3D_Protein\\RDKit_Donor\\PF00201_taxonomy33090_1.pdb.png'

<Figure size 1000x1000 with 0 Axes>

# fsdf